# AnomalyCLIP fixed-perturbation evaluation
Attach the attack ZIP dataset, target datasets, this evaluator, and the official AnomalyCLIP checkout/checkpoints as Kaggle inputs. This notebook does not regenerate attacks. It automatically calibrates clean image-F1 and pixel-F1 thresholds before adversarial evaluation and saves them in `thresholds.json`.

In [ ]:
from pathlib import Path
import json, subprocess, sys

EVALUATOR_ROOT = Path('/kaggle/input/fixed-perturbation-evaluator')  # edit
ANOMALYCLIP_ROOT = Path('/kaggle/input/anomalyclip/AnomalyCLIP')     # edit
ATTACKS_ROOT = Path('/kaggle/input/object-agnostic-attacks')         # edit
MVTEC_ROOT = Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection')  # edit
VISA_ROOT = Path('/kaggle/input/visa/VisA_20220922')                 # edit
OUTPUT_ROOT = Path('/kaggle/working/fixed_perturbation_results')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{EVALUATOR_ROOT}[anomalyclip]'], check=True)

In [ ]:
config = {
    'attacks_root': str(ATTACKS_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'model': 'anomalyclip',
    'mvtec_root': str(MVTEC_ROOT),
    'visa_root': str(VISA_ROOT),
    'targets': ['mvtec', 'visa'],
    'scopes': ['per_dataset', 'per_category', 'per_image'],
    'prompt_modes': ['frozen_prompt', 'learnable_prompt'],
    'model_kwargs_by_target': {
        'mvtec': {
            'repository': str(ANOMALYCLIP_ROOT),
            'checkpoint': str(ANOMALYCLIP_ROOT / 'checkpoints/9_12_4_multiscale/epoch_15.pth'),
            'download_root': '/kaggle/working/clip-cache',
        },
        'visa': {
            'repository': str(ANOMALYCLIP_ROOT),
            'checkpoint': str(ANOMALYCLIP_ROOT / 'checkpoints/9_12_4_multiscale_visa/epoch_15.pth'),
            'download_root': '/kaggle/working/clip-cache',
        },
    },
    'device': 'cuda', 'batch_size': 2, 'image_size': 518,
    'pixel_threshold_modes': ['fixed_0_5', 'image_f1', 'clean_pixel_f1'],
    'verify_checksums': True, 'save_predictions': False, 'overwrite': False,
}
config_path = Path('/kaggle/working/anomalyclip.json')
config_path.write_text(json.dumps(config, indent=2))
config_path

In [ ]:
subprocess.run([sys.executable, '-m', 'fpeval', '--config', str(config_path)], check=True)
threshold_path = OUTPUT_ROOT / 'anomalyclip' / 'thresholds.json'
thresholds = json.loads(threshold_path.read_text())
print('Automatically calibrated and frozen thresholds:', threshold_path)
print(json.dumps(thresholds, indent=2)[:4000])
subprocess.run(['zip', '-qr', '/kaggle/working/anomalyclip_results.zip', str(OUTPUT_ROOT / 'anomalyclip')], check=True)
print('/kaggle/working/anomalyclip_results.zip')